# PyCaret Experiments

This notebook executes all PyCaret experiments used in this thesis.

Configuration:
- Datasets: Breast Cancer, Wine, Titanic
- Seeds: 42, 123, 2026
- 5-fold Stratified Cross-Validation
- 10-minute search budget
- Macro F1 used for model selection
- Evaluation on an external untouched test set

In [10]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\souha\Downloads\Human Centered AutoML Thesis


In [11]:
import time
from pathlib import Path

import pandas as pd

from pycaret.classification import ClassificationExperiment

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [12]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

PROCESSED_ROOT = PROJECT_ROOT / "Processed"
RESULTS_ROOT = PROJECT_ROOT / "Results"

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

DATASETS = [
    "breast_cancer",
    "wine",
    "titanic"
]

SEEDS = [42, 123, 2026]

TIME_BUDGET_MINUTES = 10
CV_FOLDS = 5

In [13]:
print(PROCESSED_ROOT)
print(PROCESSED_ROOT.exists())

print(RESULTS_ROOT)
print(RESULTS_ROOT.exists())

c:\Users\souha\Downloads\Human Centered AutoML Thesis\Processed
True
c:\Users\souha\Downloads\Human Centered AutoML Thesis\Results
True


In [14]:
print(PROCESSED_ROOT.resolve())
print(PROCESSED_ROOT.exists())

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

DATASETS = [
    "breast_cancer",
    "wine",
    "titanic"
]

SEEDS = [42, 123, 2026]

TIME_BUDGET_MINUTES = 10
CV_FOLDS = 5

C:\Users\souha\Downloads\Human Centered AutoML Thesis\Processed
True


In [15]:
def run_pycaret_experiment(dataset_name, seed):
    """
    Run one PyCaret experiment using:
    - externally preprocessed data
    - 5-fold stratified cross-validation
    - a 10-minute model-comparison budget
    - Macro F1 for model selection
    - one final evaluation on the untouched test set
    """

    print(f"\nRunning PyCaret: {dataset_name} | seed {seed}")

    folder = PROCESSED_ROOT / dataset_name / f"seed_{seed}"

    # Load the exact train/test split created previously
    X_train = pd.read_csv(folder / "X_train.csv")
    X_test = pd.read_csv(folder / "X_test.csv")

    y_train = pd.read_csv(folder / "y_train.csv").squeeze("columns")
    y_test = pd.read_csv(folder / "y_test.csv").squeeze("columns")

    # PyCaret expects the target inside the DataFrame
    train_data = X_train.copy()
    train_data["target"] = y_train.to_numpy()

    test_data = X_test.copy()
    test_data["target"] = y_test.to_numpy()

    experiment = ClassificationExperiment()

    total_start = time.perf_counter()

    experiment.setup(
        data=train_data,
        target="target",

        # Give PyCaret the external untouched test set
        test_data=test_data,

        # The data is already imputed, encoded and standardized
        preprocess=False,

        fold_strategy="stratifiedkfold",
        fold=CV_FOLDS,
        fold_shuffle=True,

        session_id=seed,

        n_jobs=-1,
        verbose=False
    )

    # PyCaret's default F1 is not sufficient for our multiclass
    # methodology, so we explicitly create Macro F1.
    experiment.add_metric(
        id="macro_f1",
        name="Macro F1",
        score_func=f1_score,
        greater_is_better=True,
        multiclass=True,
        average="macro"
    )

    search_start = time.perf_counter()

    best_model = experiment.compare_models(
        sort="Macro F1",
        fold=CV_FOLDS,
        budget_time=TIME_BUDGET_MINUTES,
        turbo=True,
        errors="ignore",
        verbose=False
    )

    search_runtime_seconds = time.perf_counter() - search_start

    # Store the cross-validation leaderboard
    leaderboard = experiment.pull().copy()

    # Predict only after model selection is finished
    predictions = experiment.predict_model(
        best_model,
        data=X_test,
        verbose=False
    )

    y_pred = predictions["prediction_label"]

    total_runtime_seconds = time.perf_counter() - total_start

    result = {
        "framework": "PyCaret",
        "dataset": dataset_name,
        "seed": seed,
        "best_model": type(best_model).__name__,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_macro": precision_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        ),
        "recall_macro": recall_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        ),
        "f1_macro": f1_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        ),
        "search_runtime_seconds": search_runtime_seconds,
        "total_runtime_seconds": total_runtime_seconds
    }

    # Save the leaderboard from this individual experiment
    leaderboard_path = (
        RESULTS_ROOT
        / f"pycaret_leaderboard_{dataset_name}_seed_{seed}.csv"
    )

    leaderboard.to_csv(leaderboard_path, index=False)

    print("Best model:", result["best_model"])
    print("Test Macro F1:", round(result["f1_macro"], 4))
    print("Search runtime:", round(search_runtime_seconds, 2), "seconds")

    return result

In [16]:
print("PROCESSED_ROOT:", PROCESSED_ROOT)
print("Resolved path:", PROCESSED_ROOT.resolve())

folder = PROCESSED_ROOT / "breast_cancer" / "seed_42"

print("Experiment folder:", folder.resolve())
print("Folder exists:", folder.exists())

if folder.exists():
    print(list(folder.iterdir()))

PROCESSED_ROOT: c:\Users\souha\Downloads\Human Centered AutoML Thesis\Processed
Resolved path: C:\Users\souha\Downloads\Human Centered AutoML Thesis\Processed
Experiment folder: C:\Users\souha\Downloads\Human Centered AutoML Thesis\Processed\breast_cancer\seed_42
Folder exists: True
[WindowsPath('c:/Users/souha/Downloads/Human Centered AutoML Thesis/Processed/breast_cancer/seed_42/processed_feature_names.csv'), WindowsPath('c:/Users/souha/Downloads/Human Centered AutoML Thesis/Processed/breast_cancer/seed_42/test_row_ids.csv'), WindowsPath('c:/Users/souha/Downloads/Human Centered AutoML Thesis/Processed/breast_cancer/seed_42/train_row_ids.csv'), WindowsPath('c:/Users/souha/Downloads/Human Centered AutoML Thesis/Processed/breast_cancer/seed_42/X_test.csv'), WindowsPath('c:/Users/souha/Downloads/Human Centered AutoML Thesis/Processed/breast_cancer/seed_42/X_test_raw.csv'), WindowsPath('c:/Users/souha/Downloads/Human Centered AutoML Thesis/Processed/breast_cancer/seed_42/X_train.csv'), Wi

In [17]:
from pathlib import Path

print("Current folder:", Path.cwd())
print("Processed output will be:", (Path.cwd().parent / "Processed").resolve())

Current folder: c:\Users\souha\Downloads\Human Centered AutoML Thesis\Notebooks
Processed output will be: C:\Users\souha\Downloads\Human Centered AutoML Thesis\Processed


In [18]:
processed_splits = {}

In [21]:
print(dataset.keys())

NameError: name 'dataset' is not defined

In [22]:
print(processed_splits.keys())

dict_keys([])


In [23]:
from pathlib import Path

project_root = Path.cwd().parent
output_root = project_root / "Processed"

output_root.mkdir(parents=True, exist_ok=True)

for dataset_name, dataset_splits in processed_splits.items():
    for seed, split in dataset_splits.items():

        output_dir = output_root / dataset_name / f"seed_{seed}"
        output_dir.mkdir(parents=True, exist_ok=True)

        split["X_train"].to_csv(output_dir / "X_train.csv", index=False)
        split["X_test"].to_csv(output_dir / "X_test.csv", index=False)
        split["y_train"].to_csv(output_dir / "y_train.csv", index=False)
        split["y_test"].to_csv(output_dir / "y_test.csv", index=False)

        print(f"Saved to: {output_dir.resolve()}")

In [24]:
test_result = run_pycaret_experiment(
    dataset_name="breast_cancer",
    seed=42
)


Running PyCaret: breast_cancer | seed 42


ValueError: Invalid value for the index parameter. There are duplicate indices in the dataset. Use index=False to reset the index to RangeIndex.

In [33]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
PROCESSED_ROOT = PROJECT_ROOT / "Processed"
MODELS_ROOT = PROJECT_ROOT / "Models" / "pycaret"

MODELS_ROOT.mkdir(parents=True, exist_ok=True)

seed_folder = PROCESSED_ROOT / "titanic" / "seed_42"

X_train = pd.read_csv(seed_folder / "X_train.csv")
X_test = pd.read_csv(seed_folder / "X_test.csv")
y_train = pd.read_csv(seed_folder / "y_train.csv").squeeze("columns")
y_test = pd.read_csv(seed_folder / "y_test.csv").squeeze("columns")

In [34]:
train_data = X_train.copy()
train_data["target"] = y_train.to_numpy()

test_data = X_test.copy()
test_data["target"] = y_test.to_numpy()

train_data = train_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)

In [35]:
from pycaret.classification import setup, compare_models, save_model

experiment = setup(
    data=train_data,
    target="target",
    test_data=test_data,
    index=False,
    preprocess=False,
    fold_strategy="stratifiedkfold",
    fold=5,
    fold_shuffle=True,
    session_id=42,
    verbose=False
)

best_model = compare_models(sort="F1")

save_model(
    best_model,
    str(MODELS_ROOT / "titanic_seed_42")
)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lightgbm,Light Gradient Boosting Machine,0.7956,0.8397,0.7025,0.7500,0.7234,0.5619,0.5645,0.2860
rf,Random Forest Classifier,0.7889,0.8402,0.7150,0.7313,0.7204,0.5513,0.5540,0.1400
gbc,Gradient Boosting Classifier,0.7994,0.8524,0.6625,0.7837,0.7148,0.5623,0.5696,0.0760
lr,Logistic Regression,0.7870,0.8348,0.6875,0.7402,0.7110,0.5430,0.5457,1.4520
nb,Naive Bayes,0.7756,0.8170,0.7225,0.7019,0.7108,0.5277,0.5291,1.0540
knn,K Neighbors Classifier,0.7851,0.8218,0.6900,0.7355,0.7091,0.5395,0.5428,1.2080
ada,Ada Boost Classifier,0.7708,0.8302,0.7150,0.6987,0.7042,0.5175,0.5202,0.0600
et,Extra Trees Classifier,0.7794,0.8155,0.6875,0.7251,0.7035,0.5284,0.5309,0.1020
xgboost,Extreme Gradient Boosting,0.7803,0.8336,0.6775,0.7283,0.7010,0.5280,0.5296,0.0480
lda,Linear Discriminant Analysis,0.7765,0.8351,0.6850,0.7212,0.7006,0.5228,0.5251,0.0200


Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('placeholder', None),
                 ('trained_model',
                  LGBMClassifier(boosting_type='gbdt', class_weight=None,
                                 colsample_bytree=1.0, importance_type='split',
                                 learning_rate=0.1, max_depth=-1,
                                 min_child_samples=20, min_child_weight=0.001,
                                 min_split_gain=0.0, n_estimators=100, n_jobs=-1,
                                 num_leaves=31, objective=None, random_state=42,
                                 reg_alpha=0.0, reg_lambda=0.0, subsample=1.0,
                                 subsample_for_bin=200000, subsample_freq=0))],
          verbose=False),
 'c:\\Users\\souha\\Downloads\\Human Centered AutoML Thesis\\Models\\pycaret\\titanic_seed_42.pkl')

In [36]:
from pycaret.classification import save_model

save_model(
    best_model,
    str(MODELS_ROOT / "titanic_seed_42")
)

Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('placeholder', None),
                 ('trained_model',
                  LGBMClassifier(boosting_type='gbdt', class_weight=None,
                                 colsample_bytree=1.0, importance_type='split',
                                 learning_rate=0.1, max_depth=-1,
                                 min_child_samples=20, min_child_weight=0.001,
                                 min_split_gain=0.0, n_estimators=100, n_jobs=-1,
                                 num_leaves=31, objective=None, random_state=42,
                                 reg_alpha=0.0, reg_lambda=0.0, subsample=1.0,
                                 subsample_for_bin=200000, subsample_freq=0))],
          verbose=False),
 'c:\\Users\\souha\\Downloads\\Human Centered AutoML Thesis\\Models\\pycaret\\titanic_seed_42.pkl')

In [37]:
model_file = MODELS_ROOT / "titanic_seed_42.pkl"

print("Saved:", model_file.exists())
print("Path:", model_file.resolve())
print("Model:", best_model)

Saved: True
Path: C:\Users\souha\Downloads\Human Centered AutoML Thesis\Models\pycaret\titanic_seed_42.pkl
Model: LGBMClassifier(boosting_type='gbdt', class_weight=None, colsample_bytree=1.0,
               importance_type='split', learning_rate=0.1, max_depth=-1,
               min_child_samples=20, min_child_weight=0.001, min_split_gain=0.0,
               n_estimators=100, n_jobs=-1, num_leaves=31, objective=None,
               random_state=42, reg_alpha=0.0, reg_lambda=0.0, subsample=1.0,
               subsample_for_bin=200000, subsample_freq=0)
